# Yantra — Next Step: stops-fix re-eval + DPO Round-2 mining (guaranteed to run)

Upload to Colab → **Runtime → Run all**. Two jobs in one notebook:

1. **Re-eval (Cell 3, ~23 min on T4):** same Router V2 + GGUF as the 0.5473 run, but with fixed stop sequences (`<tool_error>`, `<reflect>` added). The last run showed `stopped_cleanly=0.1233` — the model emits `<action_end/>` correctly then hallucinates `<tool_error>…<reflect/>` cycles. Cutting generation there should lift PAS **0.5473 → ~0.64** with zero training.
2. **DPO Round-2 mining (Cell 4, ~1 min, no GPU):** mines preference pairs from the arg-failures (`exact_args=0` while router was right) into `artifacts/dpo_round2_from_eval.jsonl` in Stage-4 `pairs.jsonl` format (`prompt/chosen/rejected`), ready for the next training run.

No Unsloth, no `llama-server` — `llama-cpp-python` via `Llama()` class, same backend as the 0.5473 run so numbers are directly comparable.

In [ ]:
# @title 0 — Mount Drive & locate GGUF (preserves your 0.5473 results)
import os, sys, json, re, math, shutil, time, subprocess
from pathlib import Path
from collections import Counter

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_DIR = Path('/content/drive/MyDrive/yantra_run')
except Exception as e:
    RUN_DIR = Path('./yantra_run')
    print('Drive not mounted:', e, '→ using', RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)
ART = RUN_DIR / 'artifacts'
ART.mkdir(parents=True, exist_ok=True)
print('RUN_DIR =', RUN_DIR)
print('ART =', ART)

# Purge ONLY stale server-progress files — keep yantra_results_router_v2.json (your 0.5473)
for p in list(ART.glob('eval_progress_router_v2*.jsonl')):
    print('removing stale', p)
    try: p.unlink()
    except Exception: pass

# Show previous score for comparison
_prev = ART / 'yantra_results_router_v2.json'
if _prev.exists():
    try:
        _r = json.loads(_prev.read_text())
        print(f"Previous Router-V2 PAS = {_r.get('pas')}  summary={_r.get('summary')}")
    except Exception as e:
        print('Could not read previous results:', e)
else:
    print('No previous yantra_results_router_v2.json found (first run?)')

cands = [ART/'stage6_yantra_q4_gguf', Path('/content/stage6_yantra_q4_gguf'), Path('/content/stage6_yantra_q4')]
GGUF = None
for d in cands:
    if d.exists():
        g = sorted(d.glob('*Q4_K_M*.gguf')) or sorted(d.glob('*.gguf'))
        if g:
            GGUF = g[0]; break
if GGUF is None:
    for d in cands:
        if d.is_file():
            GGUF = d; break
if GGUF is None:
    raise FileNotFoundError(f'No GGUF found. Checked: {cands}. Run the main pipeline Stage 6 first.')
print(f'Found GGUF: {GGUF} ({GGUF.stat().st_size/1024/1024:.0f} MB)')

LOCAL_GGUF = Path('/tmp/yantra.Q4_K_M.gguf')
if str(GGUF).startswith('/content/drive'):
    if not LOCAL_GGUF.exists() or LOCAL_GGUF.stat().st_size != GGUF.stat().st_size:
        print(f'Copying GGUF to {LOCAL_GGUF} for fast inference ...')
        shutil.copy2(str(GGUF), str(LOCAL_GGUF))
        print('Copy done')
    GGUF = LOCAL_GGUF
print('Serving from:', GGUF)

if not (ART/'toolace_300.jsonl').exists():
    raise FileNotFoundError(f"{ART/'toolace_300.jsonl'} not found — run Stage 1 of main pipeline first.")
print('toolace_300.jsonl exists')


In [ ]:
# @title 1 — Install llama-cpp-python (CPU/CUDA auto, no Unsloth needed)
import importlib, subprocess, sys, os
def _need_llama():
    try:
        import llama_cpp
        print('llama_cpp', llama_cpp.__version__)
        return True
    except Exception:
        return False

if not _need_llama():
    print('Installing llama-cpp-python ... (2-4 min)')
    rc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python', '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu121']).returncode
    if rc != 0 or not _need_llama():
        print('CUDA wheel failed — trying plain pip install ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python'], check=False)
    import llama_cpp
    print('Installed llama_cpp', llama_cpp.__version__)
else:
    import llama_cpp
    print('llama_cpp already installed')

import torch
HAS_CUDA = torch.cuda.is_available()
print('HAS_CUDA =', HAS_CUDA)
if HAS_CUDA:
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# @title 2 — Router V2 + parsers + metrics (same as main pipeline Stage 6)
import json, re, math
from collections import Counter

# ---------- DTSA helpers ----------
ACTION_END = '<action_end/>'
DTSA_BIND = re.compile(r'<bind\s+tool="([^"]+)"\s*/>', re.S)
DTSA_ARGS = re.compile(r'<args>(.*?)</args>', re.S)
DTSA_PARAM = re.compile(r'<param\s+name="([^"]+)"\s*>(.*?)</param>', re.S)
def _unesc(v): return v.replace('&amp;','&').replace('&lt;','<').replace('&gt;','>')
def to_dtsa(name, arguments):
    lines = ['<bind tool="%s"/>' % name, '<args>']
    for k,v in arguments.items():
        v = '' if v is None else str(v)
        v = v.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
        lines.append('  <param name="%s">%s</param>' % (k,v))
    lines += ['</args>', ACTION_END]
    return '\n'.join(lines)
def bind_prefix(name): return f'<bind tool="{name}"/>\n'
def dtsa_args_block(name, args): return '\n'.join(to_dtsa(name,args).splitlines()[1:])
def strip_bind(text):
    m = DTSA_BIND.search(text)
    return text[m.end():] if m else text
def parse_args_block(text):
    am = DTSA_ARGS.search(text)
    if not am: return {}, False
    args = {pm.group(1): _unesc(pm.group(2).strip()) for pm in DTSA_PARAM.finditer(am.group(1))}
    ends = list(re.finditer(re.escape(ACTION_END), text))
    stopped = bool(ends) and text[ends[-1].end():].strip() == ''
    return args, stopped

# ---------- Last-bind-with-args parser (handles <param> stripped as special tokens) ----------
def parse_dtsa(text):
    binds = list(DTSA_BIND.finditer(text))
    if not binds: return None
    ends = list(re.finditer(re.escape(ACTION_END), text))
    stopped = bool(ends) and text[ends[-1].end():].strip() == ''
    for j in range(len(binds)-1, -1, -1):
        lb = binds[j]
        seg_end = binds[j+1].start() if j+1 < len(binds) else len(text)
        am = DTSA_ARGS.search(text[lb.end():seg_end])
        if not am: continue
        body = am.group(1); args = {}
        for pm in DTSA_PARAM.finditer(body):
            args[pm.group(1)] = _unesc(pm.group(2).strip())
        if not args:
            for lm in re.finditer(r'name="([^"]+)"\s*>[ \t]*(.*)', body):
                args[lm.group(1)] = _unesc(lm.group(2).strip())
        return {'tool': lb.group(1), 'args': args, 'stopped_clean': stopped}
    return {'tool': binds[0].group(1), 'args': {}, 'stopped_clean': stopped}

def parse_legacy(text):
    m = re.search(r'<function\s+name="([^"]+)"\s*>(.*?)</function>', text, re.S)
    if not m: return None
    tool = m.group(1); args = {pm.group(1): pm.group(2).strip() for pm in re.finditer(r'<param\s+name="([^"]+)"\s*>(.*?)</param>', m.group(2), re.S)}
    tail = text[m.end():].strip()
    return {'tool': tool, 'args': args, 'stopped_clean': tail == ''}

def parse_lenient(text):
    t = re.sub(r'<think>.*?</think>', '', text, flags=re.S)
    m = parse_legacy(t)
    if m: return m
    ms = list(re.finditer(r'name="([^"]+)"\s*>', t))
    if not ms: return None
    tool = ms[0].group(1); args = {}
    for i, mm in enumerate(ms[1:], 1):
        start = mm.end(); end = ms[i+1].start() if i+1 < len(ms) else len(t)
        val = t[start:end].lstrip('>').strip()
        if val: args[mm.group(1)] = val
    return {'tool': tool, 'args': args, 'stopped_clean': bool(re.search(r'</function>|'+re.escape(ACTION_END), t))}

# ---------- Metrics ----------
def _norm(a):
    if isinstance(a, str):
        s = a.strip()
        if s.lower() in ('true','false'): return s.lower()=='true'
        try: return int(s) if '.' not in s else float(s)
        except ValueError: return s
    return a
def avail_names(tools): return {t.get('function', t).get('name') for t in tools}
def evaluate_case(text, case, mode):
    tools = case['tools']; gold = case['gold']; gold0 = gold[0]
    parsed = parse_dtsa(text) if mode == 'dtsa' else parse_lenient(text)
    parseable = parsed is not None
    if not parsed: return {'parseable':0,'valid_name':0,'expected_name':0,'exact_args':0,'arg_key_overlap':0,'stopped_cleanly':0}
    valid = parsed['tool'] in avail_names(tools)
    expected = parsed['tool'] == gold0['name']
    gk = set(gold0['arguments'].keys()); pk = set(parsed['args'].keys())
    overlap = len(gk & pk)/len(gk) if gk else 1.0
    exact = (set(parsed['args'].keys()) == gk) and all(_norm(gold0['arguments'][k]) == _norm(parsed['args'][k]) for k in gk) if gk else bool(parsed)
    return {'parseable':1,'valid_name':int(valid),'expected_name':int(expected),'exact_args':int(exact),'arg_key_overlap':round(overlap,4),'stopped_cleanly':int(parsed['stopped_clean'])}
def pas(summary, recovery=0.0, multiturn=0.0):
    comps = [summary.get(k,0.0) for k in ('parseable','valid_name','expected_name','exact_args','arg_key_overlap','stopped_cleanly')] + [recovery, multiturn]
    return round(sum(comps)/len(comps), 4)
print('utils loaded')


In [ ]:
# @title 3 — Re-eval 300 cases (Router V2 + FIXED stops). ~23 min on T4.
from llama_cpp import Llama
import gc, time

# THE fix vs the 0.5473 run: stop the hallucinated <tool_error>/<reflect> tail.
# Last run's stop list was ['\n<bind','<tool_result>','<user>','</calls>'] and the model
# kept rambling past <action_end/> into fake error/reflect cycles (stopped_cleanly=0.12).
# llama.cpp drops the stop sequence from output, so text ends right after <action_end/>.
STOP = ['\n<bind', '<tool_result>', '<user>', '</calls>', '<tool_error>', '<reflect>']
print('STOP =', STOP)

print(f'Loading GGUF: {GGUF}')
n_gpu = -1 if HAS_CUDA else 0
print(f'n_gpu_layers={n_gpu} (HAS_CUDA={HAS_CUDA})')
llm = Llama(model_path=str(GGUF), n_ctx=4096, n_gpu_layers=n_gpu, verbose=False)
print('Model loaded.')

cases = [json.loads(l) for l in open(ART/'toolace_300.jsonl') if l.strip()]
print(f'Loaded {len(cases)} cases')

# ---- Router V2 (identical to 0.5473 run: IDF + char-3gram, alpha=2.0) ----
toks = lambda s: set(re.findall(r'[a-z0-9_]+', s.lower()))
_docs = [toks((t.get('function',t).get('name','')+' '+t.get('function',t).get('description',''))) for c in cases for t in c['tools']]
_df = Counter(w for d in _docs for w in d); _N = max(len(_docs),1)
_idf = lambda w: math.log(_N/(1+_df[w]))
def cgrams(s, n=3):
    s = re.sub(r'[^a-z0-9 ]','',s.lower())
    return {s[i:i+n] for i in range(max(len(s)-n+1,1))}
def route(q, tools):
    qt = toks(q); qg = cgrams(q)
    best, bn = -1e18, None
    for t in tools:
        f = t.get('function', t); nm = f.get('name') or ''
        d = toks(nm+' '+(f.get('description') or ''))
        s1 = sum(_idf(w) for w in qt & d)/(math.sqrt(sum(_idf(w) for w in d)+1e-6))
        ng = cgrams(nm)
        s2 = len(qg & ng)/(math.sqrt(len(qg)*len(ng))+1)
        if s1 + 2.0*s2 > best: best, bn = s1 + 2.0*s2, nm
    return bn

per = []       # minimal metrics per case (for results json)
per_full = []  # full records (for DPO mining in Cell 4)
t0 = time.time()
for i, case in enumerate(cases):
    query = case['query']
    gold_name = case['gold'][0]['name']
    gold_args = case['gold'][0].get('arguments', {})
    tools_json = json.dumps([t.get('function',t) for t in case['tools']], ensure_ascii=False)
    router_choice = route(query, case['tools'])
    bind = f'<bind tool="{router_choice}"/>\n'
    prompt = f"<user>{query}</user>\n<tools>{tools_json}</tools>\n<calls>{bind}"
    out = llm(prompt, max_tokens=768, temperature=0.0, stop=STOP)
    text = out['choices'][0]['text'] or ''
    m = evaluate_case(bind + text, case, mode='dtsa')
    parsed = parse_dtsa(bind + text)
    pred_tool = parsed['tool'] if parsed else None
    pred_args = parsed['args'] if parsed else {}
    per.append({'id': i, **m})
    per_full.append({
        'idx': i, 'query': query, 'tools': [t.get('function', t) for t in case['tools']],
        'gold_name': gold_name, 'gold_args': gold_args,
        'router_choice': router_choice, 'router_correct': int(router_choice == gold_name),
        'pred_tool': pred_tool, 'pred_args': pred_args,
        'raw': text[:2000], 'metrics': m,
    })
    if (i+1) % 25 == 0 or i == len(cases)-1:
        elapsed = time.time()-t0
        agg_tmp = {}
        for mm in per:
            for k,v in mm.items():
                if k == 'id': continue
                agg_tmp[k] = agg_tmp.get(k, 0.0) + v
        n = len(per); summary_tmp = {k: round(v/n,4) for k,v in agg_tmp.items()}
        print(f"  eval {i+1}/{len(cases)}  pas={pas(summary_tmp):.4f}  stopped={summary_tmp.get('stopped_cleanly',0):.3f}  {elapsed:.0f}s", flush=True)

agg = {}
for m in per:
    for k,v in m.items():
        if k == 'id': continue
        agg[k] = agg.get(k, 0.0) + v
n = len(cases); summary = {k: round(v/n,4) for k,v in agg.items()}
score = pas(summary)
res = {'summary': summary, 'pas': score, 'stop_sequences': STOP,
       'per_case': per}
out_path = ART/'yantra_results_router_v2_stopsfixed.json'
out_path.write_text(json.dumps(res, indent=2))
(ART/'yantra_eval_full_stopsfixed.jsonl').write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in per_full))
print('\n' + '='*50)
print(f'YANTRA Router-V2 (stops-fixed) PAS = {score}')
print(json.dumps(summary, indent=2))
print(f"Saved metrics to {out_path}")
print(f"Saved {len(per_full)} full records to {ART/'yantra_eval_full_stopsfixed.jsonl'} (DPO mining input)")

# Residual check: any completions still leaking error/reflect tails?
leaks = [r for r in per_full if ('<tool_error>' in r['raw'] or '<reflect' in r['raw'])]
print(f"\nResidual error/reflect leaks: {len(leaks)}/300 (want ~0)")
for r in leaks[:3]:
    print(f"  [{r['idx']}] tail: ...{repr(r['raw'][-200:])}")


In [ ]:
# @title 4 — Promote results + mine DPO Round-2 pairs (no GPU, ~1 min)
import json, re, time
from pathlib import Path
from collections import Counter

ART = RUN_DIR / 'artifacts'
NEW_RES = ART / 'yantra_results_router_v2_stopsfixed.json'
FULL = ART / 'yantra_eval_full_stopsfixed.jsonl'
EVAL = ART / 'toolace_300.jsonl'

# ---- 4a. Promote new results to official yantra_results.json (with backup) ----
if NEW_RES.exists():
    new_r = json.loads(NEW_RES.read_text())
    print(f"New PAS = {new_r['pas']}  summary={new_r['summary']}")
    old_p = ART / 'yantra_results.json'
    if old_p.exists():
        try:
            old_r = json.loads(old_p.read_text())
            print(f"Old yantra_results.json PAS = {old_r.get('pas')}")
        except Exception:
            old_r = None
        bak = ART / f"yantra_results_backup_{time.strftime('%Y%m%dT%H%M%S')}.json"
        try:
            shutil.copy2(str(old_p), str(bak))
            print(f'Backed up old results to {bak.name}')
        except Exception as e:
            print('Backup skipped:', e)
    else:
        print('No existing yantra_results.json — creating it.')
    shutil.copy2(str(NEW_RES), str(old_p))
    print(f'Promoted {NEW_RES.name} → yantra_results.json')
else:
    print(f'ERROR: {NEW_RES} not found — run Cell 3 first.')
    raise SystemExit(1)

# ---- 4b. Load full records (prefer in-memory from Cell 3, else disk) ----
try:
    _mem = per_full  # noqa: F821 — exists if Cell 3 ran in this kernel
    print(f'Using in-memory per_full ({len(_mem)} records)')
    full = _mem
except NameError:
    full = [json.loads(l) for l in open(FULL) if l.strip()]
    print(f'Loaded {len(full)} full records from {FULL.name}')

# ---- 4c. Mine pairs: router right, model args wrong → Stage-4 format ----
# Stage-4 pairs.jsonl schema: {row_idx, prompt, chosen, rejected}
# prompt uses the GOLD bind (== router bind whenever router_correct=1, so train/eval agree).
try:
    _AE = ACTION_END; _bp = bind_prefix; _dab = dtsa_args_block  # from Cell 2
except NameError:
    _AE = '<action_end/>'
    def _bp(name): return f'<bind tool="{name}"/>\n'
    def _dab(name, args):
        lines = ['<args>']
        for k, v in (args or {}).items():
            v = '' if v is None else str(v)
            lines.append('  <param name="%s">%s</param>' % (k, v))
        lines += ['</args>', _AE]
        return '\n'.join(lines)

pairs, seen, reasons = [], set(), Counter()
for r in full:
    gold_name, gold_args = r['gold_name'], r.get('gold_args', {})
    if not r.get('router_correct'):
        continue  # routing error — not an arg-training signal
    m = r.get('metrics', {})
    if m.get('exact_args') == 1:
        continue  # already correct — nothing to learn
    reason = 'arg_value' if r.get('pred_tool') == gold_name else 'tool_override'
    key = (r['query'], gold_name)
    if key in seen:
        continue
    seen.add(key)
    tools_json = json.dumps(r['tools'], ensure_ascii=False)
    prompt = f"<user>{r['query']}</user>\n<tools>{tools_json}</tools>\n<calls>" + _bp(gold_name)
    chosen = _dab(gold_name, gold_args)
    raw = (r.get('raw') or '').strip()
    cut = raw.find(_AE)
    rejected = (raw[:cut + len(_AE)] if cut != -1 else raw).strip()[:1500]
    if not rejected or rejected.strip() == chosen.strip():
        continue
    reasons[reason] += 1
    pairs.append({'row_idx': r['idx'], 'prompt': prompt, 'chosen': chosen,
                  'rejected': rejected, 'reason': reason, 'gold_name': gold_name})

out_pairs = ART / 'dpo_round2_from_eval.jsonl'
with open(out_pairs, 'w') as f:
    for p in pairs:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')

n_router_ok = sum(1 for r in full if r.get('router_correct'))
n_exact_fail = sum(1 for r in full if r.get('router_correct') and r.get('metrics', {}).get('exact_args') == 0)
print(f'\nRouter-correct: {n_router_ok}/300')
print(f'Arg-failures (router right, exact_args=0): {n_exact_fail}')
print(f'Mined DPO pairs: {len(pairs)}  reasons={dict(reasons)}')
print(f'Saved to {out_pairs}')
if pairs:
    p0 = pairs[0]
    print(f"\n--- Example pair (idx={p0['row_idx']}, reason={p0['reason']}, tool={p0['gold_name']}) ---")
    print('PROMPT (trunc):', p0['prompt'][:300].replace('\n', ' | '))
    print('CHOSEN (trunc):', p0['chosen'][:300].replace('\n', ' | '))
    print('REJECTED (trunc):', p0['rejected'][:300].replace('\n', ' | '))
    print('\nNext: feed this file into Stage-4-style DPO (same DPOTrainer config, base = stage5_rte/adapters).')
    print('Suggested: per_device_train_batch_size=2, grad_accum=4, lr=5e-5, 1 epoch (~30-60 min on T4 for ~100 pairs).')
